# IASI L2 Combined Product Reader

This notebook demonstrates how to read and explore IASI L2 Combined Product (.nat files) using the `wekeo_iasi_l3` package.

**Product contents:**
- Temperature & Humidity Profiles (101 pressure levels)
- Surface Temperature & Emissivity
- Cloud Parameters (fractional cover, top pressure/temperature, phase)
- Trace Gases (CO2, CH4, CO, N2O, O3)

In [ ]:
import os
from pathlib import Path

# Set CODA definition path (must be before importing the reader)
os.environ['CODA_DEFINITION'] = str(Path('../download').resolve())

from wekeo_iasi_l3.reader import read_iasi_l2, explore_product, IASI_L2_VARIABLES

# Point to test file
test_file = Path('../test_input/IASI_SND_02_M01_20240816222952Z_20240817001456Z_N_O_20240816233022Z')

if not test_file.exists():
    print(f"Test file not found: {test_file}")
    print("Please download IASI data first.")
else:
    print(f"Found test file: {test_file.name}")

## Available Variables

Let's see what variables we can read from the IASI L2 product:

In [ ]:
# Show all available variables
print("Available IASI L2 variables:\n")
for var, info in IASI_L2_VARIABLES.items():
    print(f"  {var:30s} - {info['description']}")

## Read IASI Data

Read the product into an xarray Dataset:

In [ ]:
# Read the IASI L2 product (default variables)
ds = read_iasi_l2(test_file)
ds

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Quick plot of surface temperature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Filter out fill values (655.35)
sfc_temp = ds.SURFACE_TEMPERATURE.values.copy()
sfc_temp[sfc_temp > 400] = np.nan

# Plot surface temperature map
ax = axes[0]
sc = ax.scatter(ds.longitude.values.flatten(), ds.latitude.values.flatten(), 
                c=sfc_temp.flatten(), s=1, cmap='RdYlBu_r', vmin=200, vmax=320)
plt.colorbar(sc, ax=ax, label='K')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Surface Temperature')

# Plot temperature profile (single footprint)
ax = axes[1]
temp_profile = ds.ATMOSPHERIC_TEMPERATURE.isel(scanline=400, footprint=60).values
pressure = ds.pressure_level.values
ax.plot(temp_profile, pressure / 100)  # Convert Pa to hPa
ax.invert_yaxis()
ax.set_yscale('log')
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Pressure (hPa)')
ax.set_title('Temperature Profile (single observation)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Trace Gases

Plot integrated trace gas columns:

In [ ]:
# Plot trace gas distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

trace_gases = [
    ('INTEGRATED_CO2', 'Total Column CO2', 'Greens'),
    ('INTEGRATED_CH4', 'Total Column CH4', 'Oranges'),
    ('INTEGRATED_CO', 'Total Column CO', 'Reds'),
    ('INTEGRATED_OZONE', 'Total Column O3', 'Purples'),
]

for ax, (var, title, cmap) in zip(axes.flatten(), trace_gases):
    data = ds[var].values.copy()
    # Filter fill values
    fill_val = data.max()
    data[data >= fill_val * 0.99] = np.nan
    
    sc = ax.scatter(ds.longitude.values.flatten(), ds.latitude.values.flatten(),
                    c=data.flatten(), s=1, cmap=cmap)
    plt.colorbar(sc, ax=ax, label=ds[var].attrs.get('units', ''))
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(title)

plt.tight_layout()
plt.show()

## Read Specific Variables

You can also select which variables to read:

In [ ]:
# Read only specific variables for faster loading
ds_clouds = read_iasi_l2(
    test_file,
    variables=[
        'EARTH_LOCATION',
        'FRACTIONAL_CLOUD_COVER',
        'CLOUD_TOP_PRESSURE',
        'CLOUD_TOP_TEMPERATURE',
        'NUMBER_CLOUD_FORMATIONS',
    ]
)

print("Cloud-focused dataset:")
print(ds_clouds)
print()
print(f"Total cloud cover (first formation):")
print(f"  Mean: {ds_clouds.FRACTIONAL_CLOUD_COVER.isel(cloud_formation=0).mean().values:.2%}")
print(f"  Max:  {ds_clouds.FRACTIONAL_CLOUD_COVER.isel(cloud_formation=0).max().values:.2%}")